In [ ]:
import os
import random
import scipy.io as sio
import numpy as np
import cv2
import pywt
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.signal import butter, filtfilt, resample
from torch.utils.data import Dataset, Subset, DataLoader
from torchvision import transforms, models
from torchvision.models import VGG16_Weights
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def butter_bandpass_filter(data, lowcut=500.0, highcut=4900.0, fs=51200.0, order=3):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band', analog=False)
    y = filtfilt(b, a, data)
    return y

In [ ]:

class TargetDataset(Dataset):
    def __init__(self, data_list, labels, original_fs, target_fs, segment_length=400, transform=None):
        self.samples = []
        self.transform = transform
        self.target_fs = target_fs

        for signal, label in zip(data_list, labels):
            filtered_signal = butter_bandpass_filter(signal, lowcut=500.0, highcut=4900.0, fs=original_fs)
            
            if target_fs < original_fs:
                num_samples = int(len(filtered_signal) * (target_fs / original_fs))
                resampled_signal = resample(filtered_signal, num_samples)
            else:
                resampled_signal = filtered_signal
                
            num_segments = len(resampled_signal) // segment_length
            for i in range(num_segments):
                start_idx = i * segment_length
                self.samples.append((resampled_signal[start_idx:start_idx + segment_length], label))

    def __len__(self):
        return len(self.samples)

    def generate_cwt(self, signal):
        fs = self.target_fs  
        fc = 1.0      
        frequencies = np.linspace(fs/2, 50, 128) 
        scales = (fc * fs) / frequencies
        wavelet = 'cmor5.0-1.0' 
        
        coefficients, _ = pywt.cwt(signal, scales, wavelet, sampling_period=1/fs)
        amplitude = np.power(np.abs(coefficients), 2)
        
        amp_min, amp_max = amplitude.min(), amplitude.max()
        if amp_max > amp_min:
            normalized_amp = (amplitude - amp_min) / (amp_max - amp_min)
        else:
            normalized_amp = amplitude
            
        img_8bit = np.uint8(normalized_amp * 255)
        img_color = cv2.applyColorMap(img_8bit, cv2.COLORMAP_JET)
        img_resized = cv2.resize(img_color, (224, 224))
        return cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx):
        signal_segment, label = self.samples[idx]
        image_np = self.generate_cwt(signal_segment)
        
        if self.transform:
            image_tensor = self.transform(image_np)
        else:
            transform_default = transforms.Compose([transforms.ToTensor()])
            image_tensor = transform_default(image_np)
            
        return image_tensor, torch.tensor(label, dtype=torch.long)



In [ ]:
def extract_acoustic_signal(mat_obj):
    if hasattr(mat_obj, 'dtype') and mat_obj.dtype.names is not None:
        for name in mat_obj.dtype.names:
            res = extract_acoustic_signal(mat_obj[name])
            if res is not None: return res
    elif isinstance(mat_obj, np.void):
        for name in mat_obj.dtype.names:
            res = extract_acoustic_signal(mat_obj[name])
            if res is not None: return res
    elif isinstance(mat_obj, np.ndarray):
        if mat_obj.size > 1000:
            return mat_obj.flatten()
        for item in mat_obj.flat:
            res = extract_acoustic_signal(item)
            if res is not None: return res
    return None

def load_kaist_data(data_dir, duration_seconds=10):
    data_list = []
    labels = []
    original_fs = 51200
    points_to_extract = duration_seconds * original_fs
    
    file_map = {
        '0Nm_Normal.mat': 0, 
        '0Nm_BPFI_03.mat': 1,     
        '0Nm_BPFI_10.mat': 2,    
        '0Nm_BPFO_03.mat': 3,    
        '0Nm_BPFO_10.mat': 4      
    }
    
    for filename, label in file_map.items():
        file_path = os.path.join(data_dir, filename)

        mat_data = sio.loadmat(file_path)
        signal = None
        
        if 'Signal' in mat_data:
            signal = extract_acoustic_signal(mat_data['Signal'])
        else:
            for key in mat_data:
                if not key.startswith('__'):
                    signal = extract_acoustic_signal(mat_data[key])
                    if signal is not None:
                        break
        
        if signal is not None:
            actual_extract = min(points_to_extract, len(signal))
            data_list.append(signal[:actual_extract])
            labels.append(label)
        else:
            print(f" failed  {filename}")
            
    return data_list, labels

kaist_dir = "/kaggle/input/datasets/onkarraskar/kaist-data/Kaist_acoustic"
print("KAIST Acoustic data files...")
kaist_signals, kaist_labels = load_kaist_data(kaist_dir, duration_seconds=10)

print("Initializing dataset ")
kaist_dataset = TargetDataset(
    data_list=kaist_signals, 
    labels=kaist_labels, 
    original_fs=51200.0, 
    target_fs=10000.0, 
    segment_length=400
)

class_indices = {i: [] for i in range(5)}

for idx, (_, label) in enumerate(kaist_dataset.samples):
    class_indices[label].append(idx)

train_indices, val_indices, test_indices = [], [], []

train_per_class = 30   
val_per_class = 15     
test_per_class = 140   
gap = 5                

for label, indices in class_indices.items():
    train_block = indices[0 : train_per_class]
    val_block   = indices[train_per_class + gap : train_per_class + gap + val_per_class]
    test_start  = train_per_class + gap + val_per_class + gap
    test_block  = indices[test_start : test_start + test_per_class]

    train_indices.extend(train_block)
    val_indices.extend(val_block)
    test_indices.extend(test_block)

train_dataset = Subset(kaist_dataset, train_indices)
val_dataset = Subset(kaist_dataset, val_indices)
test_dataset = Subset(kaist_dataset, test_indices)

batch_size = 32
kaist_train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
kaist_val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
kaist_test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


print(f" Split Complete:")
print(f"  --> Training Samples:   {len(train_dataset)}")
print(f"  --> Validation Samples: {len(val_dataset)}")
print(f"  --> Testing Samples:    {len(test_dataset)}")


In [ ]:
transfer_model = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
in_features = transfer_model.classifier[6].in_features

transfer_model.classifier[6] = nn.Linear(in_features, 10)
transfer_model.load_state_dict(torch.load('/kaggle/input/datasets/onkarraskar/cwru-best-weight/cwru_pretrained_vgg16 (1).pth', map_location=device))
print("Successfully loaded pre-trained CWRU baseline weights.")

transfer_model.classifier[6] = nn.Linear(in_features, 5)
transfer_model.classifier[2] = nn.Dropout(p=0.6)
transfer_model.classifier[5] = nn.Dropout(p=0.6)
transfer_model = transfer_model.to(device)

for param in transfer_model.features.parameters():
    param.requires_grad = False
for param in transfer_model.classifier.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer_stage1 = optim.Adam(transfer_model.classifier.parameters(), lr=1e-4)

num_epochs_stage1 = 20
best_val_loss = float('inf')
patience = 5
epochs_no_improve = 0

print("\nStarting Stage 1: Retraining Classifier...")
for epoch in range(num_epochs_stage1):
    transfer_model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in kaist_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_stage1.zero_grad()
        outputs = transfer_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_stage1.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100. * correct / total
        
    transfer_model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in kaist_val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = transfer_model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    avg_val_loss = val_loss / total_val
    val_acc = 100. * correct_val / total_val
    
    print(f"Epoch [{epoch+1:02d}/{num_epochs_stage1:02d}] | Train Loss: {running_loss/total:.4f} - Train Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.2f}%")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(transfer_model.state_dict(), 'kaist_stage1_vgg16.pth')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"   --> [!] Early stopping triggered at epoch {epoch+1}.")
            break

print("\nStage 1 Complete.")

In [ ]:
transfer_model.load_state_dict(torch.load('/kaggle/working/kaist_stage1_vgg16.pth', map_location=device))

for param in transfer_model.features[24:].parameters():
    param.requires_grad = True

optimizer_stage2 = optim.Adam(filter(lambda p: p.requires_grad, transfer_model.parameters()), lr=1e-5)
best_val_loss = float('inf')
epochs_no_improve = 0
patience = 5

print("\nStarting Stage 2: Fine-Tuning Conv5...")
for epoch in range(15):
    transfer_model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in kaist_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_stage2.zero_grad()
        outputs = transfer_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_stage2.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100. * correct / total
        
    transfer_model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in kaist_val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = transfer_model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    avg_val_loss = val_loss / total_val
    val_acc = 100. * correct_val / total_val
    
    print(f"Epoch [{epoch+1:02d}/15] | Train Loss: {running_loss/total:.4f} - Train Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.2f}%")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(transfer_model.state_dict(), 'kaist_stage2_vgg16.pth')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("   --> [!] Early stopping triggered.")
            break


In [ ]:
print("==========STAGE 2 COMPLETE. INITIATING TESTING==========")


transfer_model.load_state_dict(torch.load('/kaggle/working/kaist_stage2_vgg16.pth', map_location=device))
transfer_model.eval()

all_preds, all_labels = [], []
test_correct, test_loss, total_test = 0, 0.0, 0

with torch.no_grad():
    for inputs, labels in kaist_test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = transfer_model(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        
        total_test += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

avg_test_loss = test_loss / total_test
final_test_acc = 100. * test_correct / total_test

print(f"TEST LOSS: {avg_test_loss:.4f}")
print(f"TEST ACCURACY (KAIST Acoustic): {final_test_acc:.2f}% \n")

cm = confusion_matrix(all_labels, all_preds)
class_names = ['Healthy', 'Inner 0.3mm', 'Inner 1.0mm', 'Outer 0.3mm', 'Outer 1.0mm']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'KAIST Acoustic Test Confusion Matrix (Acc: {final_test_acc:.2f}%)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))